# GSE221156 human pancreatic islets: no diabetes (ND) versus type 2 diabetes (T2D)

This notebook follows the standard Seurat workflow (Satija lab tutorials): read the data, build a Seurat object, look at the quality-control
plots, filter, and save. On top of that it does the work this dataset needs: match every library to its donor, take each donor's BMI, and sort donors into **lean** and **not lean**.

**The comparison:** ND (17 donors) against T2D (17 donors). The 14 pre-diabetic (PD) donors are left out by default (`groups_to_keep`).

**What to know before running**
* **12 of the 54 libraries are pools of two donors** (titles like `Islet70/71_MS19034`). Three T2D donors (Islet71, 84, 119) exist **only** in pools.
  The paper separated the donors with Demuxlet, which needs genotypes that are not on GEO, so those donors' **cells** can be used only if you get a barcode-to-donor table (section 5).
  Without it you have **17 ND and 14 T2D donors**. Their **BMI is still known**, because GEO lists a value per donor for pooled libraries.
* **This is pancreatic islets, not adipose tissue.**
* **BMI differs between ND and T2D** and **chemistry (V2 or V3 reagent kit) is uneven between them**, so both are covariates for later tests.

I tested the code on small made-up data only. Read each table before trusting it.

## 1. Setup

**Step:** Load the packages and set the paths and options. Change only this cell.

In [ ]:
library(Seurat)
library(Matrix)
library(ggplot2)

tar_file <- "C:/Users/user/Downloads/GSE221156_RAW.tar"
raw_dir  <- "C:/Users/user/Downloads/GSE221156_RAW"
out_dir  <- "GSE221156_results"

groups_to_keep <- c("ND", "T2D")   # add "PD" for the pre-diabetic donors
lean_cutoff    <- 25               # BMI below this = lean
obese_cutoff   <- 30               # BMI from this = obese (25 to 29.9 = overweight)

demux_file <- ""                   # barcode-to-donor table for pooled libraries (section 5); "" if you do not have it
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

## 2. Find the files

**Step:** Unpack the archive if needed, then list the libraries. Each library has three files: `_matrix.mtx.gz`, `_features.tsv.gz` and `_barcodes.tsv.gz`.

In [ ]:
if (length(list.files(raw_dir, pattern = "_matrix\\.mtx\\.gz$")) == 0) {
  dir.create(raw_dir, showWarnings = FALSE, recursive = TRUE)
  untar(tar_file, exdir = raw_dir)
}

matrix_files <- list.files(raw_dir, pattern = "_matrix\\.mtx\\.gz$", full.names = TRUE, recursive = TRUE)
libraries <- data.frame(file_prefix = sub("_matrix\\.mtx\\.gz$", "", matrix_files))
libraries$sample_id  <- basename(libraries$file_prefix)
libraries$gsm        <- sub("_.*$", "", libraries$sample_id)           # GSM6846488
libraries$library_id <- sub("^GSM[0-9]+_", "", libraries$sample_id)    # MS17001
nrow(libraries)

*What you should see:* 54 libraries.

## 3. Sample information from GEO

**Step:** Download the GEO sample records once (title and a text field with sex, BMI, age, HbA1c, chemistry and so on) and save them, so later runs work offline.

In [ ]:
geo_file <- file.path(out_dir, "geo_sample_metadata.csv")

if (file.exists(geo_file)) {
  geo <- read.csv(geo_file, stringsAsFactors = FALSE)
} else {
  if (!requireNamespace("GEOquery", quietly = TRUE)) BiocManager::install("GEOquery")
  gse <- GEOquery::getGEO("GSE221156", GSEMatrix = FALSE)
  records <- GEOquery::GSMList(gse)
  geo <- data.frame(
    gsm             = names(records),
    title           = sapply(records, function(x) GEOquery::Meta(x)$title[1]),
    characteristics = sapply(records, function(x) paste(GEOquery::Meta(x)$characteristics_ch1, collapse = " || "))
  )
  write.csv(geo, geo_file, row.names = FALSE)
}
head(geo[, c("gsm", "title")], 3)

**Step:** Split the text field into one column per item (`bmi`, `sex`, `chemistry`, ...).

In [ ]:
split_characteristics <- function(text) {
  items  <- strsplit(text, " \\|\\| ")[[1]]
  values <- trimws(sub("^[^:]*:", "", items))
  names(values) <- tolower(gsub("[^A-Za-z0-9]+", "_", sub(":.*$", "", items)))
  as.list(values)
}

parsed <- lapply(geo$characteristics, split_characteristics)
for (field in unique(unlist(lapply(parsed, names)))) {
  geo[[field]] <- sapply(parsed, function(x) if (is.null(x[[field]])) NA_character_ else x[[field]])
}
names(geo)[names(geo) == "library_id"] <- "geo_library_id"   # keep it apart from our own library_id

libraries <- merge(libraries, geo, by = "gsm", all.x = TRUE, sort = FALSE)
colnames(geo)

*What you should see:* Columns such as chemistry, disease_state, sex, bmi, race, hba1c, age and center.

## 4. Libraries and donors

**Step:** Which donor is in each library? A title like `Islet28` is one donor. A title like `Islet70/71` is a **pool** of two.

In [ ]:
donor_numbers <- regmatches(libraries$title, gregexpr("(?<=Islet)[0-9]+|(?<=/)[0-9]+", libraries$title, perl = TRUE))
libraries$donors <- sapply(donor_numbers, function(x) paste0("Islet", x, collapse = "+"))
libraries$pooled <- sapply(donor_numbers, length) > 1
libraries$donor  <- ifelse(libraries$pooled, NA, libraries$donors)

table(libraries$pooled)

*What you should see:* 42 single-donor and 12 pooled libraries.

**Step:** The donor groups, copied from the GEO series page (17 ND, 14 PD, 17 T2D).

In [ ]:
nd_ids  <- c(29, 34, 37, 38, 42, 45, 47, 54, 57, 59, 61, 68, 73, 80, 104, 107, 123)
pd_ids  <- c(40, 50, 53, 55, 56, 58, 62, 63, 70, 85, 89, 101, 118, 127)
t2d_ids <- c(28, 30, 31, 32, 33, 39, 41, 44, 48, 52, 60, 71, 84, 91, 126, 119, 128)

donor_groups <- data.frame(
  donor = paste0("Islet", c(nd_ids, pd_ids, t2d_ids)),
  group = rep(c("ND", "PD", "T2D"), c(length(nd_ids), length(pd_ids), length(t2d_ids)))
)
donor_groups$group_label <- c(ND = "No diabetes", PD = "Pre-diabetes", T2D = "Type 2 diabetes")[donor_groups$group]
libraries$group <- donor_groups$group[match(libraries$donor, donor_groups$donor)]   # NA for pools

**Step:** Which donors of the groups you keep have **no library of their own**? They sit only inside pools.

In [ ]:
donor_groups$own_library <- donor_groups$donor %in% libraries$donor
table(donor_groups$group, donor_groups$own_library)

only_in_pools <- donor_groups[!donor_groups$own_library & donor_groups$group %in% groups_to_keep, ]
only_in_pools

*What you should see:* All 17 ND donors have a library. For T2D, 14 do and Islet71, Islet84 and Islet119 do not.

## 5. Barcodes of pooled libraries

In a single-donor library every cell belongs to that donor. In a **pooled** library each barcode must be assigned to one of the two donors.
The paper did this with Demuxlet, using donor SNP genotypes that are **not** in the GEO files, and I cannot recover the assignment from the counts alone.
Where to look for a barcode table: the authors' Zenodo record 14656366 and the CELLxGENE collection named in the paper, or the corresponding author.

If you have a table with the **library id, the barcode and the donor**, set `demux_file` in section 1 and the column names below. A donor value starting with `DBL` is a doublet.
Without a table, pooled libraries are **left out and counted, never guessed**.

**Step:** Read the barcode table if there is one, and keep only assignments that make sense (the donor must be one of the two in that library).

In [ ]:
demux_library_col <- "library_id"
demux_barcode_col <- "barcode"
demux_donor_col   <- "donor"

clean_barcode <- function(x) sub("-1$", "", sub("^.*_", "", as.character(x)))
clean_donor   <- function(x) ifelse(grepl("^DBL", x), "Doublet", paste0("Islet", gsub("[^0-9]", "", x)))

demux <- NULL
if (demux_file != "") {
  table_in <- read.delim(demux_file, sep = ifelse(grepl("\\.csv", demux_file), ",", "\t"), stringsAsFactors = FALSE)
  demux <- data.frame(
    library_id = sub("^GSM[0-9]+_", "", table_in[[demux_library_col]]),
    barcode    = clean_barcode(table_in[[demux_barcode_col]]),
    donor      = clean_donor(table_in[[demux_donor_col]])
  )
  allowed <- strsplit(libraries$donors[match(demux$library_id, libraries$library_id)], "+", fixed = TRUE)
  ok <- demux$donor == "Doublet" | mapply(function(d, a) d %in% a, demux$donor, allowed)
  cat("assignments that do not match the library's two donors (dropped):", sum(!ok), "\n")
  demux$donor[!ok] <- "Inconsistent"
  demux$key <- paste(demux$library_id, demux$barcode)
} else {
  cat("No barcode table: pooled libraries will be left out.\n")
}

## 6. Donor table: BMI, lean or not lean

**Step:** Start the donor table from the **single-donor libraries**: sex, age, HbA1c, BMI, race and islet centre come straight from the GEO record.

In [ ]:
to_number <- function(x) suppressWarnings(as.numeric(x))

single <- libraries[!libraries$pooled, ]
donors <- data.frame(
  donor  = single$donor,
  sex    = single$sex,
  age    = to_number(single$age),
  hba1c  = to_number(single$hba1c),
  bmi    = to_number(single$bmi),
  race   = single$race,
  center = single$center
)
donors$bmi_source <- "GEO sample record"
donors <- merge(donor_groups, donors, by = "donor", all.x = TRUE, sort = FALSE)
sum(is.na(donors$bmi))

*What you should see:* Six donors have no BMI yet: the PD and T2D donors who are only in pools (70, 71, 84, 85, 118, 119).

**Step:** For pooled libraries GEO stores **one value per donor, joined by `/` in title order** (`Islet70/71` has `bmi: 32.9/31.7`). Split them.

In [ ]:
split_pair <- function(x, n) {
  parts <- strsplit(x, "/", fixed = TRUE)[[1]]
  if (length(parts) == n) parts else rep(NA_character_, n)
}

pooled <- libraries[libraries$pooled, ]
pooled_donors <- NULL
for (i in seq_len(nrow(pooled))) {
  ids <- strsplit(pooled$donors[i], "+", fixed = TRUE)[[1]]
  n   <- length(ids)
  one <- data.frame(
    donor  = ids,
    state  = split_pair(pooled$disease_state[i], n),
    sex    = split_pair(pooled$sex[i], n),
    age    = to_number(split_pair(pooled$age[i], n)),
    hba1c  = to_number(split_pair(pooled$hba1c[i], n)),
    bmi    = to_number(split_pair(pooled$bmi[i], n)),
    race   = split_pair(pooled$race[i], n),
    center = split_pair(pooled$center[i], n)
  )
  pooled_donors <- rbind(pooled_donors, one)
}
pooled_donors <- pooled_donors[!duplicated(pooled_donors$donor), ]   # two libraries per pool: keep one row
pooled_donors[, c("donor", "state", "sex", "age", "bmi")]

**Step:** **Check the order before using these values.** Check 1: the disease state of each donor in a pool must match the donor lists. Check 2: donors who have both their own library and a pool must give the same BMI and age.

In [ ]:
pooled_donors$state_group <- ifelse(grepl("type ?2", pooled_donors$state, ignore.case = TRUE), "T2D",
                             ifelse(grepl("pre", pooled_donors$state, ignore.case = TRUE), "PD", "ND"))
list_group <- donor_groups$group[match(pooled_donors$donor, donor_groups$donor)]
states_match <- all(pooled_donors$state_group == list_group)

both <- merge(donors[, c("donor", "bmi", "age")], pooled_donors[, c("donor", "bmi", "age")], by = "donor", suffixes = c("_own", "_pool"))
both <- both[!is.na(both$bmi_own) & !is.na(both$bmi_pool), ]
values_match <- nrow(both) > 0 && all(abs(both$bmi_own - both$bmi_pool) < 0.05) && all(both$age_own == both$age_pool)

print(both)
cat("Check 1, disease states match the donor lists:", states_match, "\n")
cat("Check 2, BMI and age match on shared donors:   ", values_match, "\n")

*What you should see:* Both checks TRUE, with six donors compared (47, 48, 57, 58, 59, 60).

**Step:** If both checks pass, fill the six donors who only have a pooled record.

In [ ]:
if (states_match && values_match) {
  needs_fill <- is.na(donors$bmi) & donors$donor %in% pooled_donors$donor
  from_pool  <- match(donors$donor[needs_fill], pooled_donors$donor)
  for (column in c("sex", "age", "hba1c", "bmi", "race", "center")) {
    donors[[column]][needs_fill] <- pooled_donors[[column]][from_pool]
  }
  donors$bmi_source[needs_fill] <- "GEO pooled-library record"
  cat("filled:", donors$donor[needs_fill], "\n")
} else {
  cat("The checks failed: pooled values were NOT used.\n")
}

**Step:** Sort donors into lean, overweight and obese, and into lean or not lean.

In [ ]:
donors$bmi_class   <- cut(donors$bmi, c(-Inf, lean_cutoff, obese_cutoff, Inf), right = FALSE, labels = c("Lean", "Overweight", "Obese"))
donors$bmi_class   <- as.character(donors$bmi_class)
donors$lean_status <- ifelse(donors$bmi < lean_cutoff, "Lean", "Not lean")
donors <- donors[order(donors$group, as.integer(sub("Islet", "", donors$donor))), ]
write.csv(donors, file.path(out_dir, "donor_table_BMI.csv"), row.names = FALSE)

table(donors$group, donors$lean_status, useNA = "ifany")

*What you should see:* ND: 5 lean and 12 not lean. T2D: 1 lean (Islet84) and 16 not lean.

**Step:** Check the labels and compare the two groups. HbA1c should fit the group, and BMI, age and sex should be similar if ND was matched to T2D.

In [ ]:
kept <- donors[donors$group %in% groups_to_keep, ]

aggregate(hba1c ~ group, data = kept, FUN = function(x) round(c(mean = mean(x), min = min(x), max = max(x)), 2))
aggregate(cbind(bmi, age) ~ group, data = kept, FUN = median)
table(kept$group, kept$sex)

wilcox.test(bmi ~ group, data = kept, exact = FALSE)$p.value
wilcox.test(age ~ group, data = kept, exact = FALSE)$p.value

*What you should see:* HbA1c mean about 5.2 (ND) and 7.6 (T2D), as in the paper. BMI differs (p about 0.02), age does not.

## 7. Build the Seurat object

**Step:** Which libraries to read: the single-donor libraries of the kept groups, plus the pooled libraries that hold a kept donor with no library of their own, **if** you have a barcode table.

In [ ]:
missing_donors <- only_in_pools$donor
holds_missing  <- sapply(strsplit(libraries$donors, "+", fixed = TRUE), function(x) any(x %in% missing_donors))

read_these <- (!libraries$pooled & libraries$group %in% groups_to_keep) | (libraries$pooled & holds_missing & !is.null(demux))
to_read <- libraries[read_these, ]
cat("libraries to read:", nrow(to_read), "\n")
if (is.null(demux)) cat("not read (no barcode table):", libraries$library_id[libraries$pooled & holds_missing], "\n")

*What you should see:* 31 libraries.

**Step:** Read each library and join them into one genes x cells matrix. Cell names are `library_barcode`, so they cannot collide.

In [ ]:
read_library <- function(prefix, library_id) {
  counts   <- as(readMM(paste0(prefix, "_matrix.mtx.gz")), "CsparseMatrix")
  features <- read.delim(paste0(prefix, "_features.tsv.gz"), header = FALSE, stringsAsFactors = FALSE)
  barcodes <- readLines(gzfile(paste0(prefix, "_barcodes.tsv.gz")))
  colnames(counts) <- paste(library_id, barcodes, sep = "_")
  list(counts = counts, genes = features[[2]], gene_ids = features[[1]])
}

all_counts <- list()
for (i in seq_len(nrow(to_read))) {
  one <- read_library(to_read$file_prefix[i], to_read$library_id[i])
  if (i == 1) gene_names <- one$genes
  stopifnot(identical(gene_names, one$genes))     # same genes in the same order in every library
  all_counts[[i]] <- one$counts
}
counts <- do.call(cbind, all_counts)
rownames(counts) <- make.unique(gene_names)
rm(all_counts); gc()
dim(counts)

*What you should see:* 36,601 genes and about 219,000 cells (without the pooled libraries).

**Step:** Make the cell metadata: library, donor, group, BMI and the other covariates. In pooled libraries the donor comes from the barcode table.

In [ ]:
cell_info <- data.frame(row.names = colnames(counts))
cell_info$library_id <- sub("_[^_]+$", "", colnames(counts))
cell_info$barcode    <- sub("^.*_", "", colnames(counts))
from_library <- match(cell_info$library_id, libraries$library_id)

cell_info$donor          <- libraries$donor[from_library]
cell_info$library_pooled <- libraries$pooled[from_library]
cell_info$chemistry      <- libraries$chemistry[from_library]      # V2 or V3 reagent kit

if (!is.null(demux)) {
  in_pool <- which(cell_info$library_pooled)
  hit <- match(paste(cell_info$library_id[in_pool], clean_barcode(cell_info$barcode[in_pool])), demux$key)
  cell_info$donor[in_pool] <- ifelse(is.na(hit), "Unassigned", demux$donor[hit])
}

# donor-level information, copied to every cell of that donor
from_donor <- match(cell_info$donor, donors$donor)
for (column in c("group", "group_label", "sex", "age", "hba1c", "bmi", "bmi_class", "lean_status", "race", "center", "bmi_source")) {
  cell_info[[column]] <- donors[[column]][from_donor]
}

# keep only cells whose donor is known and in the groups you chose
keep <- cell_info$group %in% groups_to_keep
table(keep)

**Step:** Create the Seurat object, as in the Seurat tutorial. `percent.mt` is the percentage of counts from mitochondrial genes.

In [ ]:
obj <- CreateSeuratObject(counts = counts[, keep], meta.data = cell_info[keep, ], project = "GSE221156")
obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
obj$group_lean <- paste(obj$group, obj$lean_status, sep = "_")      # for example ND_Lean, T2D_Not lean
rm(counts); gc()
obj

**Step:** Check what is in the object. Target: 17 ND and 17 T2D donors.

In [ ]:
donors_in_object <- unique(obj@meta.data[, c("donor", "group", "lean_status", "chemistry")])
table(donors_in_object$group)
table(donors_in_object$group, donors_in_object$lean_status)
table(donors_in_object$group, donors_in_object$chemistry)
setdiff(donors$donor[donors$group %in% groups_to_keep], donors_in_object$donor)    # donors with no cells in the object

*What you should see:* 17 ND and 14 T2D without the barcode table. The last line lists Islet71, Islet84 and Islet119. V2 is uneven: more T2D than ND donors.

## 8. Quality control
Following the Seurat tutorial and your guide: look at the metrics per library first, then filter, and keep a log of what was removed. The paper's order was ambient RNA, then doublets, then cut-offs.

| Step | Paper | Here |
|---|---|---|
| Ambient RNA | SoupX (needs the empty droplets of raw matrices) | **DecontX** (GEO has filtered matrices only) |
| Doublets | Scrublet and DoubletFinder | **scDblFinder** |
| Cut-offs | genes ≥ 1000, UMIs > 1000, mito ≤ 40% | the same, in the settings below |

Beta and alpha cells have more mitochondrial reads than other islet cells, so the paper used 40% overall and then **20% for non-beta cell types** after annotation (apply that to the endothelial cells later).

**Step:** QC settings. Change the cut-offs after you have looked at the plots and the table below.

In [ ]:
min_features <- 1000     # genes detected per cell
min_counts   <- 1000     # UMIs per cell
max_mito     <- 40       # percent mitochondrial
qc_dir <- file.path(out_dir, "qc")
dir.create(qc_dir, showWarnings = FALSE, recursive = TRUE)
set.seed(1)

**Step:** The three standard QC plots, one violin per donor (as in the Seurat tutorial).

In [ ]:
violins <- VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), group.by = "donor", pt.size = 0, ncol = 1) & NoLegend()
violins
ggsave(file.path(qc_dir, "violin_per_donor.png"), violins, width = 12, height = 12, dpi = 120)

*What you should see:* Look for a donor far from the rest, and for V2 libraries (donors Islet28 to 41) sitting at a different depth.

**Step:** The tutorial's scatter plots, on a random 20,000 cells so they draw quickly (**for the picture only**; every cell is used for the numbers).

In [ ]:
plot_cells <- sample(colnames(obj), min(20000, ncol(obj)))
FeatureScatter(obj, "nCount_RNA", "percent.mt",  cells = plot_cells, group.by = "chemistry") +
FeatureScatter(obj, "nCount_RNA", "nFeature_RNA", cells = plot_cells, group.by = "chemistry")

**Step:** What each possible cut-off would remove, by group and by chemistry. A cut-off that removes far more cells from one column than another would itself create a difference.

In [ ]:
meta <- obj@meta.data
would_remove <- data.frame(
  genes_below_500  = meta$nFeature_RNA < 500,
  genes_below_1000 = meta$nFeature_RNA < 1000,
  genes_below_1400 = meta$nFeature_RNA < 1400,
  umis_below_1000  = meta$nCount_RNA < 1000,
  mito_above_15    = meta$percent.mt > 15,
  mito_above_20    = meta$percent.mt > 20,
  mito_above_40    = meta$percent.mt > 40,
  mito_above_50    = meta$percent.mt > 50
)
percent_removed <- function(by) sapply(split(would_remove, by), function(d) round(100 * colMeans(d), 1))
percent_removed(meta$group)
percent_removed(meta$chemistry)

*What you should see:* Percent of cells removed, one column per group (then per chemistry).

### Ambient RNA (DecontX, one library at a time)

**Step:** Estimate each cell's contamination and write the corrected counts into a second assay, `decontX`. **The original counts stay in `RNA`.** The result is saved, so a restart does not repeat the 1 to 3 minutes per library.

In [ ]:
decontx_file <- file.path(qc_dir, "decontx.rds")

if (file.exists(decontx_file)) {
  decontx <- readRDS(decontx_file)
} else if (requireNamespace("celda", quietly = TRUE)) {
  raw_counts <- LayerData(obj, assay = "RNA", layer = "counts")
  corrected  <- list()
  contamination <- c()
  for (lib in unique(obj$library_id)) {
    cells_in_lib <- colnames(obj)[obj$library_id == lib]
    result <- celda::decontX(raw_counts[, cells_in_lib], verbose = FALSE)
    corrected[[lib]]  <- drop0(round(result$decontXcounts))
    contamination     <- c(contamination, setNames(result$contamination, cells_in_lib))
    cat(lib, "done\n")
  }
  decontx <- list(counts = do.call(cbind, corrected)[, colnames(obj)], contamination = contamination[colnames(obj)])
  saveRDS(decontx, decontx_file)
} else {
  decontx <- NULL
  cat("celda is not installed, so the ambient step is skipped: BiocManager::install('celda')\n")
}

if (!is.null(decontx)) {
  obj$decontx_contamination <- decontx$contamination
  obj[["decontX"]] <- CreateAssay5Object(counts = decontx$counts)
  aggregate(decontx_contamination ~ group + chemistry, data = obj@meta.data, FUN = median)
}

*What you should see:* Median contamination per group and chemistry (islet data often 5 to 20%).

### Doublets (scDblFinder, one library at a time)

**Step:** Flag likely doublets in each library from the original counts. Saved after the first run.

In [ ]:
doublet_file <- file.path(qc_dir, "doublets.csv")
obj$doublet_class <- "not_tested"

if (file.exists(doublet_file)) {
  saved <- read.csv(doublet_file, row.names = 1)
  obj$doublet_class <- saved[colnames(obj), "class"]
} else if (requireNamespace("scDblFinder", quietly = TRUE)) {
  raw_counts <- LayerData(obj, assay = "RNA", layer = "counts")
  calls <- setNames(rep("not_tested", ncol(obj)), colnames(obj))
  for (lib in unique(obj$library_id)) {
    cells_in_lib <- colnames(obj)[obj$library_id == lib]
    sce <- SingleCellExperiment::SingleCellExperiment(assays = list(counts = raw_counts[, cells_in_lib]))
    sce <- scDblFinder::scDblFinder(sce, verbose = FALSE)
    calls[cells_in_lib] <- as.character(sce$scDblFinder.class)
    cat(lib, ":", sum(calls[cells_in_lib] == "doublet"), "doublets\n")
  }
  obj$doublet_class <- calls[colnames(obj)]
  write.csv(data.frame(class = calls), doublet_file)
} else {
  cat("scDblFinder is not installed, so doublets are not removed: BiocManager::install('scDblFinder')\n")
}
table(obj$group, obj$doublet_class)

*What you should see:* About 2 to 8% doublets per library for 10x.

### Filter and keep a log

**Step:** Filter with `subset()`, as in the tutorial. First record which cells fail which test.

In [ ]:
meta <- obj@meta.data
meta$low_genes <- meta$nFeature_RNA < min_features
meta$low_umis  <- meta$nCount_RNA < min_counts
meta$high_mito <- meta$percent.mt > max_mito
meta$doublet   <- meta$doublet_class == "doublet"
meta$kept      <- !(meta$low_genes | meta$low_umis | meta$high_mito | meta$doublet)

flags <- data.frame(library_id = meta$library_id, donor = meta$donor, group = meta$group, chemistry = meta$chemistry,
                    cells = 1, low_genes = meta$low_genes, low_umis = meta$low_umis, high_mito = meta$high_mito,
                    doublet = meta$doublet, kept = meta$kept)
exclusion_log <- aggregate(. ~ library_id + donor + group + chemistry, data = flags, FUN = sum)
exclusion_log$percent_kept <- round(100 * exclusion_log$kept / exclusion_log$cells, 1)
exclusion_log <- exclusion_log[order(exclusion_log$percent_kept), ]
write.csv(exclusion_log, file.path(qc_dir, "exclusion_log.csv"), row.names = FALSE)
head(exclusion_log, 10)

*What you should see:* The ten libraries that lose the most cells.

**Step:** Compare retention between ND and T2D. QC should not remove very different shares of cells from the two groups.

In [ ]:
round(100 * tapply(meta$kept, meta$group, mean), 1)
round(100 * tapply(meta$kept, list(meta$group, meta$chemistry), mean), 1)

exclusion_log_two <- exclusion_log[exclusion_log$group %in% c("ND", "T2D"), ]
wilcox.test(percent_kept ~ group, data = exclusion_log_two, exact = FALSE)$p.value

*What you should see:* Similar percentages across rows; a small p-value means the filters treat the groups differently.

**Step:** Apply the filter.

In [ ]:
obj_qc <- subset(obj, subset = nFeature_RNA >= min_features & nCount_RNA >= min_counts & percent.mt <= max_mito & doublet_class != "doublet")
c(before = ncol(obj), after = ncol(obj_qc))
table(obj_qc$group, obj_qc$chemistry)
head(sort(table(obj_qc$donor)), 5)       # donors with the fewest cells left

**Step:** Check the plots after filtering, and save the filtered object.

In [ ]:
violins_after <- VlnPlot(obj_qc, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), group.by = "donor", pt.size = 0, ncol = 1) & NoLegend()
violins_after
ggsave(file.path(qc_dir, "violin_per_donor_after_qc.png"), violins_after, width = 12, height = 12, dpi = 120)

saveRDS(obj_qc, file.path(out_dir, "GSE221156_ND_T2D_seurat_QC.rds"))
write.csv(obj_qc@meta.data, file.path(out_dir, "cell_metadata_after_QC.csv"))

## Next steps
Normalise, find variable genes, scale, run PCA, choose the number of PCs with an elbow plot, cluster and run UMAP (the tutorial's remaining steps), integrating across donors with chemistry as a batch variable.
Then name the clusters, find the endothelial cells (the paper's marker is `PLVAP`), apply the 20% mito limit to them, and compare **T2D with ND per donor** (pseudobulk with edgeR), with BMI, age, sex and chemistry as covariates where the numbers allow.

**Limits.** Donor numbers, not cell numbers, set the power. About 90 endothelial cells per donor on average means ERG may be barely detected in many donors; keep read counts next to every ERG result.
Run the pseudobulk test once on `RNA` and once on `decontX` and compare. This is pancreatic islets, not adipose tissue.